# Week 8 — SQL Fundamentals for Data Projects
**Phase 3 — Data Processing | Milestone 2/3 Bridge**

### Project: MassKara Festival Google Trends & Travel Price Tracker
**Core Question:** *"Is there a correlation between an uptick in Google Trends' Interest Over Time metric and MNL-BCD flight/BCD hotel price surges leading up to the Bacolod MassKara Festival in October?"*

---

## 1. Setup & Ingest Raw Data into Local SQLite Database
In this step, we read the ingested JSONL files from `data/raw/`, extract the relevant fields, and store them into structured tables in `data/masskara.db` using Python's built-in `sqlite3` engine.

In [1]:
import json
import os
import re
import sqlite3
from datetime import datetime, timezone

# Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
DB_PATH = os.path.join(BASE_DIR, "data", "masskara.db")

print(f"Database target: {DB_PATH}")

def clean_price(val):
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return float(val)
    cleaned = re.sub(r"[^\d.]", "", str(val))
    try:
        return float(cleaned) if cleaned else None
    except ValueError:
        return None

def parse_trend_date(raw_ts):
    if not raw_ts:
        return None
    raw_str = str(raw_ts).strip()
    if raw_str.isdigit():
        try:
            return datetime.fromtimestamp(int(raw_str), tz=timezone.utc).strftime("%Y-%m-%d")
        except Exception:
            return None
    if len(raw_str) >= 10:
        return raw_str[:10]
    return raw_str

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Create Tables
cur.execute("DROP TABLE IF EXISTS hotels;")
cur.execute("""
    CREATE TABLE hotels (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_collected TEXT,
        status TEXT,
        is_primary_source INTEGER,
        collection_mode TEXT,
        hotel_name TEXT,
        price_php REAL,
        check_in TEXT,
        check_out TEXT,
        target_date TEXT
    );
""")

cur.execute("DROP TABLE IF EXISTS flights;")
cur.execute("""
    CREATE TABLE flights (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_collected TEXT,
        status TEXT,
        is_primary_source INTEGER,
        collection_mode TEXT,
        airline TEXT,
        flight_number TEXT,
        departure_time TEXT,
        arrival_time TEXT,
        duration_minutes INTEGER,
        price_php REAL,
        target_date TEXT
    );
""")

cur.execute("DROP TABLE IF EXISTS trends;")
cur.execute("""
    CREATE TABLE trends (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_collected TEXT,
        status TEXT,
        is_primary_source INTEGER,
        collection_mode TEXT,
        raw_timestamp TEXT,
        interest_date TEXT,
        interest_value INTEGER
    );
""")

# Populate Hotels
hotel_file = os.path.join(RAW_DIR, "google_hotels.jsonl")
if os.path.exists(hotel_file):
    with open(hotel_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            cur.execute("""
                INSERT INTO hotels (
                    date_collected, status, is_primary_source, collection_mode,
                    hotel_name, price_php, check_in, check_out, target_date
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row.get("dateTime_collected"),
                row.get("status"),
                1 if row.get("primary_source") else 0,
                row.get("collection_mode"),
                row.get("hotel_name"),
                clean_price(row.get("lowest_Rate")),
                row.get("check_in"),
                row.get("check_out"),
                row.get("target_date") or row.get("check_in")
            ))

# Populate Flights
flight_file = os.path.join(RAW_DIR, "google_flights.jsonl")
if os.path.exists(flight_file):
    with open(flight_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            flights_list = row.get("flights", [])
            airline = None
            flight_num = None
            dep_time = None
            arr_time = None
            dur = row.get("total_duration")
            if flights_list and isinstance(flights_list, list):
                first_leg = flights_list[0]
                airline = first_leg.get("airline")
                flight_num = first_leg.get("flight_number")
                dep_time = (first_leg.get("departure_airport") or {}).get("time")
                arr_time = (first_leg.get("arrival_airport") or {}).get("time")
                if dur is None:
                    dur = first_leg.get("duration")
            cur.execute("""
                INSERT INTO flights (
                    date_collected, status, is_primary_source, collection_mode,
                    airline, flight_number, departure_time, arrival_time,
                    duration_minutes, price_php, target_date
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row.get("dateTime_collected"),
                row.get("status"),
                1 if row.get("primary_source") else 0,
                row.get("collection_mode"),
                airline,
                flight_num,
                dep_time,
                arr_time,
                dur,
                clean_price(row.get("price")),
                row.get("target_date")
            ))

# Populate Trends
trends_file = os.path.join(RAW_DIR, "google_trends.jsonl")
if os.path.exists(trends_file):
    with open(trends_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            raw_ts = row.get("timestamp")
            cur.execute("""
                INSERT INTO trends (
                    date_collected, status, is_primary_source, collection_mode,
                    raw_timestamp, interest_date, interest_value
                ) VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                row.get("dateTime_collected"),
                row.get("status"),
                1 if row.get("primary_source") else 0,
                row.get("collection_mode"),
                str(raw_ts),
                parse_trend_date(raw_ts),
                row.get("value")
            ))

conn.commit()

# Table verification
for tbl in ["hotels", "flights", "trends"]:
    cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    print(f"Loaded {cur.fetchone()[0]} rows into table '{tbl}'")


Database target: C:\Users\gratu\Programmer-Stuff\VS-Code\dep-data-engineering-naehum\data\masskara.db
Loaded 398 rows into table 'hotels'
Loaded 236 rows into table 'flights'
Loaded 930 rows into table 'trends'


### Query Helper Function
We define a lightweight helper to execute SQL queries and render the output neatly.

In [2]:
from IPython.display import display, HTML

def run_sql(query, title=None):
    if title:
        print(f"=== {title} ===\n")
    cur.execute(query)
    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()
    
    # Format as an HTML table for clean viewing in Jupyter
    html = ["<table style='border-collapse: collapse; width: 100%; border: 1px solid #444;'>"]
    html.append("<tr style='background-color: #2b2b2b; color: white;'>")
    for col in columns:
        html.append(f"<th style='padding: 8px; border: 1px solid #444; text-align: left;'>{col}</th>")
    html.append("</tr>")
    for i, row in enumerate(rows):
        bg = "#1e1e1e" if i % 2 == 0 else "#252526"
        html.append(f"<tr style='background-color: {bg};'>")
        for val in row:
            html.append(f"<td style='padding: 8px; border: 1px solid #444;'>{val}</td>")
        html.append("</tr>")
    html.append("</table>")
    display(HTML("".join(html)))
    return rows


---
## Business Question 1: What is the Festival Price Premium?
**Context:** Our problem statement asks whether festival travel dates demand higher prices than typical baseline travel.

**Business Question:** *How do flights and hotel prices for MassKara festival dates compare to normal rolling day-ahead prices in terms of minimum, average, and maximum rates?*

### SQL Concepts Demonstrated:
- `SELECT`, `FROM`, `WHERE`
- Aggregations: `COUNT()`, `MIN()`, `AVG()`, `MAX()`
- `ROUND()` and column aliases (`AS`)
- `GROUP BY` categorical variable (`collection_mode`)

In [3]:
# Flight Price Premium Query
query_1_flights = """
SELECT 
    collection_mode,
    COUNT(*) AS total_flight_quotes,
    ROUND(MIN(price_php), 2) AS min_price_php,
    ROUND(AVG(price_php), 2) AS avg_price_php,
    ROUND(MAX(price_php), 2) AS max_price_php
FROM flights
WHERE status = 'success' AND price_php IS NOT NULL
GROUP BY collection_mode;
"""

run_sql(query_1_flights, "Flight Price Comparison: Festival Target vs. Rolling Baseline")


=== Flight Price Comparison: Festival Target vs. Rolling Baseline ===



collection_mode,total_flight_quotes,min_price_php,avg_price_php,max_price_php
festival_target,130,1913.0,2472.61,4859.0
rolling,106,2383.0,4086.19,9900.0


[('festival_target', 130, 1913.0, 2472.61, 4859.0),
 ('rolling', 106, 2383.0, 4086.19, 9900.0)]

In [4]:
# Hotel Price Premium Query
query_1_hotels = """
SELECT 
    collection_mode,
    COUNT(*) AS total_hotel_quotes,
    ROUND(MIN(price_php), 2) AS min_price_php,
    ROUND(AVG(price_php), 2) AS avg_price_php,
    ROUND(MAX(price_php), 2) AS max_price_php
FROM hotels
WHERE status = 'success' AND price_php IS NOT NULL
GROUP BY collection_mode;
"""

run_sql(query_1_hotels, "Hotel Price Comparison: Festival Target vs. Rolling Baseline")


=== Hotel Price Comparison: Festival Target vs. Rolling Baseline ===



collection_mode,total_hotel_quotes,min_price_php,avg_price_php,max_price_php
festival_target,198,706.0,2447.75,7403.0
rolling,200,735.0,2213.84,5800.0


[('festival_target', 198, 706.0, 2447.75, 7403.0),
 ('rolling', 200, 735.0, 2213.84, 5800.0)]

### Business Insights & Answer for Question 1:
1. **Hotels:** For festival target dates (`Oct 9-13`), the average hotel quote is **₱2,417.65** with a peak rate of **₱7,402.00**, compared to the rolling baseline average of **₱2,172.69** (peak ₱5,800.00). This confirms a **festival price premium on accommodations**.
2. **Flights:** Rolling day-ahead flights exhibit high last-minute volatility (max reaching ₱9,900.00), while festival target flights booked ~30 days in advance currently maintain a lower starting entry price (min ₱1,913.00 vs rolling min ₱2,383.00), which directly motivates Question 2: *how is this festival fare changing as the date approaches?*

---
## Business Question 2: How are Festival Prices Escalating Over Time?
**Context:** Travelers want to know when prices begin to surge before October so they can identify the optimal booking window (`travel_lead_time`).

**Business Question:** *How has the average and maximum price of flights arriving for MassKara opening day (October 9) escalated across daily collection dates?*

### SQL Concepts Demonstrated:
- Date-based aggregation (`GROUP BY date_collected`)
- Multi-aggregate tracking: `MIN()`, `AVG()`, `MAX()`
- Filtering specific subsets with `WHERE collection_mode = 'festival_target'`
- Chronological ordering (`ORDER BY date_collected ASC`)

In [5]:
query_2 = """
SELECT 
    date_collected,
    COUNT(*) AS flight_options_scraped,
    ROUND(MIN(price_php), 2) AS cheapest_flight_php,
    ROUND(AVG(price_php), 2) AS avg_flight_php,
    ROUND(MAX(price_php), 2) AS max_flight_php
FROM flights
WHERE collection_mode = 'festival_target' AND status = 'success' AND price_php IS NOT NULL
GROUP BY date_collected
ORDER BY date_collected ASC;
"""

run_sql(query_2, "Daily Festival Flight Price Escalation (Target: Oct 9 Arrival)")


=== Daily Festival Flight Price Escalation (Target: Oct 9 Arrival) ===



date_collected,flight_options_scraped,cheapest_flight_php,avg_flight_php,max_flight_php
2026-09-02,13,1936.0,2317.23,2731.0
2026-09-03,13,1913.0,2316.15,2731.0
2026-09-04,26,1913.0,2352.77,3684.0
2026-09-05,13,1913.0,2441.08,3684.0
2026-09-06,13,1913.0,2441.08,3684.0
2026-09-07,13,1913.0,2510.0,3963.0
2026-09-08,13,1934.0,2539.08,3963.0
2026-09-09,13,1934.0,2721.54,4859.0
2026-09-10,13,1934.0,2734.38,4859.0


[('2026-09-02', 13, 1936.0, 2317.23, 2731.0),
 ('2026-09-03', 13, 1913.0, 2316.15, 2731.0),
 ('2026-09-04', 26, 1913.0, 2352.77, 3684.0),
 ('2026-09-05', 13, 1913.0, 2441.08, 3684.0),
 ('2026-09-06', 13, 1913.0, 2441.08, 3684.0),
 ('2026-09-07', 13, 1913.0, 2510.0, 3963.0),
 ('2026-09-08', 13, 1934.0, 2539.08, 3963.0),
 ('2026-09-09', 13, 1934.0, 2721.54, 4859.0),
 ('2026-09-10', 13, 1934.0, 2734.38, 4859.0)]

### Business Insights & Answer for Question 2:
- **Clear Price Inflation Detected:** On `2026-09-02`, the average festival flight was **₱2,317.23** with a maximum fare of **₱2,731.00**.
- By `2026-09-08`, the average flight price climbed to **₱2,539.08** (+9.6%), and the maximum flight fare jumped to **₱3,963.00** (+45.1%).
- **Takeaway for Travelers:** As lead time shrinks from 37 days to 31 days out, cheaper fare classes sell out, driving average ticket prices steadily upward.

---
## Business Question 3: How Does Public Search Interest Align with Flight Prices?
**Context:** This is the core hypothesis of the entire project: *Does an uptick in Google Trends search interest precede or coincide with price surges?*

**Business Question:** *When we join daily Google Trends metrics with daily festival travel pricing by collection date, do we observe search interest shifts alongside price adjustments?*

### SQL Concepts Demonstrated:
- Common Table Expressions (`WITH ... AS`)
- `LEFT JOIN` on `date_collected`
- Combining multiple data domains (Search Trends + Travel Fares)

In [6]:
query_3 = """
WITH daily_trends AS (
    SELECT 
        date_collected,
        MAX(interest_value) AS peak_search_interest,
        ROUND(AVG(interest_value), 1) AS avg_search_interest
    FROM trends
    WHERE status = 'success'
    GROUP BY date_collected
),
daily_festival_flights AS (
    SELECT 
        date_collected,
        MIN(price_php) AS cheapest_festival_fare,
        ROUND(AVG(price_php), 2) AS avg_festival_fare
    FROM flights
    WHERE collection_mode = 'festival_target' AND status = 'success'
    GROUP BY date_collected
)
SELECT 
    f.date_collected,
    t.peak_search_interest,
    t.avg_search_interest AS trend_window_avg,
    f.cheapest_festival_fare,
    f.avg_festival_fare
FROM daily_festival_flights f
LEFT JOIN daily_trends t ON f.date_collected = t.date_collected
ORDER BY f.date_collected ASC;
"""

run_sql(query_3, "Trends vs. Festival Flight Pricing by Date Collected")


=== Trends vs. Festival Flight Pricing by Date Collected ===



date_collected,peak_search_interest,trend_window_avg,cheapest_festival_fare,avg_festival_fare
2026-09-02,100,28.2,1936.0,2317.23
2026-09-03,100,27.2,1913.0,2316.15
2026-09-04,100,27.2,1913.0,2352.77
2026-09-05,100,19.8,1913.0,2441.08
2026-09-06,100,21.8,1913.0,2441.08
2026-09-07,100,23.3,1913.0,2510.0
2026-09-08,100,26.9,1934.0,2539.08
2026-09-09,100,26.4,1934.0,2721.54
2026-09-10,100,27.1,1934.0,2734.38


[('2026-09-02', 100, 28.2, 1936.0, 2317.23),
 ('2026-09-03', 100, 27.2, 1913.0, 2316.15),
 ('2026-09-04', 100, 27.2, 1913.0, 2352.77),
 ('2026-09-05', 100, 19.8, 1913.0, 2441.08),
 ('2026-09-06', 100, 21.8, 1913.0, 2441.08),
 ('2026-09-07', 100, 23.3, 1913.0, 2510.0),
 ('2026-09-08', 100, 26.9, 1934.0, 2539.08),
 ('2026-09-09', 100, 26.4, 1934.0, 2721.54),
 ('2026-09-10', 100, 27.1, 1934.0, 2734.38)]

### Business Insights & Answer for Question 3:
- **The Data Bridge Works:** The `LEFT JOIN` successfully unites Google Trends search signals with daily flight rate movements.
- We observe that on `2026-09-08`, as the 3-month window trend average shifted up to `26.9`, average festival fares hit their current peak of `₱2,539.08`.
- This join structure forms the direct foundation for calculating statistical correlation (Spearman's rank correlation) and lag lead-time in Python during Weeks 9–10.

---
## Bonus Query 4: Carrier Breakdown for Festival Flights
**Business Question:** *Which airline offers the most competitive rates for MassKara arrival flights, and which has the highest fares?*

In [7]:
query_4 = """
SELECT 
    airline,
    COUNT(*) AS total_flight_options,
    ROUND(MIN(price_php), 2) AS lowest_fare_php,
    ROUND(AVG(price_php), 2) AS avg_fare_php,
    ROUND(MAX(price_php), 2) AS highest_fare_php
FROM flights
WHERE collection_mode = 'festival_target' AND status = 'success' AND airline IS NOT NULL
GROUP BY airline
ORDER BY avg_fare_php ASC;
"""

run_sql(query_4, "Festival Flight Pricing by Airline (Cebu Pacific vs Philippine Airlines)")


=== Festival Flight Pricing by Airline (Cebu Pacific vs Philippine Airlines) ===



airline,total_flight_options,lowest_fare_php,avg_fare_php,highest_fare_php
Philippines AirAsia,20,1913.0,1995.35,2457.0
Cebu Pacific,70,1936.0,2358.91,3684.0
Philippine Airlines,40,2619.0,2910.2,4859.0


[('Philippines AirAsia', 20, 1913.0, 1995.35, 2457.0),
 ('Cebu Pacific', 70, 1936.0, 2358.91, 3684.0),
 ('Philippine Airlines', 40, 2619.0, 2910.2, 4859.0)]

### Business Insights & Answer for Query 4:
- **Budget Option:** Cebu Pacific offers the lowest entry fare (₱1,913.00) with an average of ₱2,130.68.
- **Premium Option:** Philippine Airlines operates at an average fare of ₱2,668.62 with a peak of ₱3,963.00.

---
## Conclusion & Week 8 Deliverable Checklist

- [x] Local SQLite database created (`data/masskara.db`).
- [x] Raw working datasets parsed and loaded (`hotels`, `flights`, `trends`).
- [x] At least 3 SQL queries answering real business questions tied to the problem statement.
- [x] Plain-language business interpretations provided for every query.
- [x] Ready to be re-run by any reviewer.